<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 26 · Algorithmic Trading in the Real World

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the Chapter 26 closing discussion in an interactive
format. It works through implementation shortfall, break-even arithmetic, and
hedge-fund benchmark comparisons using the local monthly dataset.


### How to Use This Notebook
- Run the cells from top to bottom the first time.
- The setup cell switches into the project root and adds `code/` to
  `sys.path`.
- The notebook defines its analytical helpers itself instead of importing
  the chapter module.


### Notebook Setup
Move to the project root first so that the notebook can reuse the same
relative paths and local packages as the chapter scripts.


In [ ]:
from pathlib import Path  # filesystem paths
import os  # working-directory handling
import sys  # local package imports

PROJECT_ROOT = Path.cwd().resolve()  # current notebook location
if not (PROJECT_ROOT / "data" / "eod_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # support launches from notebooks/
os.chdir(PROJECT_ROOT)  # switch to the book project root

CODE_PATH = PROJECT_ROOT / "code"
if str(CODE_PATH) not in sys.path:
    sys.path.insert(0, str(CODE_PATH))

Path.cwd()


## Imports and Local Dataset Path
Import the numerical libraries and define the local hedge-fund dataset path
used throughout the notebook.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

HF_DATA = PROJECT_ROOT / "data" / "hf_data.csv"
HF_DATA


## Practical Helpers
Define the friction, return, and benchmark-diagnostic helpers directly in the
notebook.


In [ ]:
# Helper functions for the chapter metrics and benchmark diagnostics.

def implementation_shortfall_example(
    gross_alpha: float,
    transaction_costs: float,
    slippage: float,
    infrastructure: float,
) -> pd.Series:
    net_alpha = gross_alpha - transaction_costs - slippage - infrastructure
    retention = net_alpha / gross_alpha if gross_alpha != 0.0 else np.nan
    return pd.Series(
        {
            "gross_alpha": gross_alpha,
            "transaction_costs": transaction_costs,
            "slippage": slippage,
            "infrastructure": infrastructure,
            "net_alpha": net_alpha,
            "retention_ratio": retention,
        }
    )


def break_even_hit_rate(
    avg_gain: float,
    avg_loss: float,
    transaction_cost_per_trade: float,
) -> float:
    effective_gain = avg_gain - transaction_cost_per_trade
    effective_loss = avg_loss + transaction_cost_per_trade
    return effective_loss / (effective_gain + effective_loss)


def load_hf_data(path: Path = HF_DATA) -> pd.DataFrame:
    data = pd.read_csv(path)
    data.columns = data.columns.str.lower()
    data = data.rename(columns={"hf_index": "hedge_fund"})
    return (
        data.assign(date=pd.to_datetime(data["date"]))
        .set_index("date")
        .sort_index()
    )


def annualized_return(returns: pd.Series) -> float:
    return float((1.0 + returns).prod() ** (12.0 / len(returns)) - 1.0)


def annualized_volatility(returns: pd.Series) -> float:
    return float(returns.std(ddof=0) * np.sqrt(12.0))


def max_drawdown(returns: pd.Series) -> float:
    equity = (1.0 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1.0
    return float(drawdown.min())


def sharpe_ratio(returns: pd.Series) -> float:
    ann_vol = annualized_volatility(returns)
    if ann_vol == 0.0:
        return float("nan")
    return annualized_return(returns) / ann_vol


def sortino_ratio(returns: pd.Series) -> float:
    downside = returns[returns < 0.0].std(ddof=0) * np.sqrt(12.0)
    if pd.isna(downside) or downside == 0.0:
        return float("nan")
    return annualized_return(returns) / float(downside)


def build_hf_summary_table(data: pd.DataFrame) -> pd.DataFrame:
    rets = data.copy()
    rets["6040"] = 0.6 * rets["spy"] + 0.4 * rets["ief"]
    spy_vol = annualized_volatility(rets["spy"])
    hf_vol = annualized_volatility(rets["hedge_fund"])
    leverage = spy_vol / hf_vol
    rets["hf_lev"] = leverage * rets["hedge_fund"]

    rows = []
    for column in ["hedge_fund", "spy", "ief", "6040", "hf_lev"]:
        series = rets[column]
        rows.append(
            {
                "series": column,
                "ann_return": annualized_return(series),
                "ann_vol": annualized_volatility(series),
                "cum_return": float((1.0 + series).prod() - 1.0),
                "max_dd": max_drawdown(series),
                "sharpe": sharpe_ratio(series),
                "sortino": sortino_ratio(series),
            }
        )
    return pd.DataFrame(rows).set_index("series")


def benchmark_diagnostics(data: pd.DataFrame, column: str) -> pd.Series:
    series = data[column]
    benchmark = data["spy"]
    beta = np.cov(series, benchmark, ddof=0)[0, 1] / np.var(benchmark, ddof=0)
    tracking_error = (series - benchmark).std(ddof=0) * np.sqrt(12.0)
    active_ann_return = annualized_return(series) - annualized_return(benchmark)
    info_ratio = (
        active_ann_return / tracking_error
        if tracking_error > 0.0
        else np.nan
    )
    up_mask = benchmark > 0.0
    down_mask = benchmark < 0.0
    return pd.Series(
        {
            "beta": float(beta),
            "corr": float(series.corr(benchmark)),
            "tracking_error": float(tracking_error),
            "active_ann_return": float(active_ann_return),
            "information_ratio": float(info_ratio),
            "up_capture": float(
                series[up_mask].mean() / benchmark[up_mask].mean()
            ),
            "down_capture": float(
                series[down_mask].mean() / benchmark[down_mask].mean()
            ),
        }
    )


def calendar_return_table(data: pd.DataFrame) -> pd.DataFrame:
    rets = data.copy()
    rets["6040"] = 0.6 * rets["spy"] + 0.4 * rets["ief"]
    leverage = annualized_volatility(rets["spy"]) / annualized_volatility(
        rets["hedge_fund"]
    )
    rets["hf_lev"] = leverage * rets["hedge_fund"]
    return (
        (1.0 + rets[["hedge_fund", "spy", "6040", "hf_lev"]])
        .groupby(rets.index.year)
        .prod()
        - 1.0
    )


## A Simple Implementation-Shortfall Calculation
Start with a compact friction budget and observe how transaction costs,
slippage, and infrastructure reduce a gross expected edge.


In [ ]:
shortfall = implementation_shortfall_example(
    gross_alpha=0.08,
    transaction_costs=0.015,
    slippage=0.010,
    infrastructure=0.005,
)
shortfall.round(4)


## Break-Even Hit Rate
Compute the win probability required for a strategy to break even once a
round-trip trading friction is included.


In [ ]:
break_even_hit_rate(
    avg_gain=0.012,
    avg_loss=0.010,
    transaction_cost_per_trade=0.001,
)


## Loading the Hedge-Fund Comparison Dataset
The local dataset contains aligned monthly returns for the Barclay Hedge Fund
Index, `SPY`, and `IEF`.


In [ ]:
hf_data = load_hf_data()
hf_data.head()


## Return and Risk Summary
Build the same comparison table used in the chapter for the hedge-fund index,
`SPY`, `IEF`, a simple `60/40` mix, and the volatility-matched hedge-fund
series.


In [ ]:
summary = build_hf_summary_table(hf_data)
summary.round(4)


## Benchmark Diagnostics
Measure beta, tracking error, active return, information ratio, and capture
ratios relative to `SPY`.


In [ ]:
diagnostics_hf = benchmark_diagnostics(hf_data, "hedge_fund")
diagnostics_hf.round(4)


In [ ]:
diagnostics_6040 = benchmark_diagnostics(
    hf_data.assign(**{"6040": 0.6 * hf_data["spy"] + 0.4 * hf_data["ief"]}),
    "6040",
)
diagnostics_6040.round(4)


In [ ]:
diagnostics_hf_lev = benchmark_diagnostics(
    hf_data.assign(
        **{
            "hf_lev": (
                annualized_volatility(hf_data["spy"])
                / annualized_volatility(hf_data["hedge_fund"])
            ) * hf_data["hedge_fund"]
        }
    ),
    "hf_lev",
)
diagnostics_hf_lev.round(4)


## Calendar-Year Perspective
Aggregate the monthly returns to calendar-year returns so the benchmark and
hedge-fund trade-offs are visible across concrete market environments.


In [ ]:
calendar = calendar_return_table(hf_data)
calendar.round(4)
